# HW6: Model Predictive Control — Look Ahead, Act Once, Replan
**NTUT MuJoCo Course — Lecture 7 · 25 points + 5 bonus**

Lecture 4 designed LQR around an upright equilibrium. Here we make the finite-horizon
optimization explicit, add physical bounds, and solve a QP at every control instant.
This lesson prepares the Open Duck MPC midterm; the cart-pole input is **wheel torque**, not ZMP.

| Part | Points | Student work |
|---|---:|---|
| A | 10 | Quadratic cost, stacked dynamics, initial state and bounds |
| B | 8 | Extract the first action and implement receding-horizon feedback |
| C | 7 | Compare LQR/MPC, horizons and disturbances; explain evidence |
| D | +5 | Soft state bounds with hard input limits |

Use bounded `for` loops, do not edit TEST CELLs, and do not change fixed test tolerances.
All numerical sections run headlessly; animation is optional. This notebook is independent of completed HW3.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.linalg import expm
from IPython.display import HTML, display
for root in (Path.cwd(), *Path.cwd().parents):
    if (root/'lecture7'/'cartpole.py').exists():
        sys.path.insert(0,str(root/'lecture7')); break
else: raise FileNotFoundError('Start inside the course repository')
from cartpole import PARAMS, dynamics, rk4, linear_model, lqr
from qp import solve_qp
from visualization import plot_comparison, plot_windows, animate_cartpole, metrics
DT=.02
A,B,Ad,Bd=linear_model(DT)
Q=np.diag([1.,10.,1.,1.]); R=np.array([[1.]])
K,P=lqr(Ad,Bd,Q,R)
print('Model input: total wheel torque (Nm); state: [s, theta, s_dot, theta_dot]')

## Physical model and prediction window

Use the same reduced wheel-driven model as Lecture 4:
$$\begin{bmatrix}a&b\cos\theta\\b\cos\theta&c\end{bmatrix}
\begin{bmatrix}\ddot s\\\ddot\theta\end{bmatrix}
=\begin{bmatrix}b\dot\theta^2\sin\theta+u/r\\Mg\ell\sin\theta-u\end{bmatrix}.$$
Here $a=M+m_w+I_w/r^2$, $b=M\ell$, $c=I_b+M\ell^2$. The $-u$ term is the
reaction torque on the body. We retain Lecture 4's state ordering and upright $\theta=0$.
The provided parameters were extracted at its 0.30 m equilibrium; no teacher imports are needed.

Linearization gives $\dot x=Ax+Bu$. For exact zero-order hold,
$$\exp\left(\begin{bmatrix}A&B\\0&0\end{bmatrix}\Delta t\right)
=\begin{bmatrix}A_d&B_d\\0&1\end{bmatrix}.$$
Lecture 4 differentiated an RK4 step; here the matrix exponential discretizes the same
continuous linear model exactly. The simulated plant still uses **nonlinear RK4**.

$x_{k+i|k}$ means the prediction for time $k+i$ based on information available at $k$.
$N$ counts future control intervals; the time window is $N\Delta t$. There are $N+1$
states and $N$ controls. Predictions start from the newly measured state, not yesterday's prediction.

In [ ]:
# Given model sanity checks; independent finite differences.
eps=1e-6
A_fd=np.column_stack([(dynamics(np.eye(4)[j]*eps,0)-dynamics(-np.eye(4)[j]*eps,0))/(2*eps) for j in range(4)])
B_fd=((dynamics(np.zeros(4),eps)-dynamics(np.zeros(4),-eps))/(2*eps))[:,None]
np.testing.assert_allclose(A,A_fd,atol=1e-8)
np.testing.assert_allclose(B,B_fd,atol=1e-8)
assert np.max(np.abs(np.linalg.eigvals(Ad-Bd@K)))<1
print('Given cart-pole model checks passed')

## Part A: Derive the QP (10 pts)

**Goal.** OSQP does not know about states or inputs; it only solves
$$\min_z\ \tfrac12z^THz+f^Tz\qquad\text{s.t.}\qquad l\le Gz\le u.$$
Part A *translates* the MPC problem into `(H, f, G, lower, upper)`. Nothing is solved here;
Part B passes your arrays to `solve_qp`.

The problem to translate: minimize
$$J=\sum_{i=0}^{N-1}(x_i^TQx_i+u_i^TRu_i)+x_N^TP_fx_N,$$
subject to $x_0=\hat x_k$, $x_{i+1}=A_dx_i+B_du_i$, and symmetric state/input bounds.
$Q\succeq0$, $R\succ0$; choose $P_f=P_{\mathrm{DARE}}$ for the LQR comparison.

### Step 1: decision vector and sizes
Stack every unknown, **all states first, then all inputs**:
$$z=[x_0^T,\ldots,x_N^T,u_0^T,\ldots,u_{N-1}^T]^T.$$
$N$ controls produce $N+1$ states, because both $x_0$ and $x_N$ count.
Read $n,m$ from `B.shape` and let `nx = (N+1)*n`, `nu = N*m`.

| Quantity | Formula | Test value ($n=4$, $m=1$, $N=3$) |
|---|---|---:|
| length of $z$ (size of $H$) | `nx + nu` | 19 |
| equality rows (rows of $E$) | $(N+1)n$ | 16 |
| bound rows | `nx + nu` | 19 |
| shape of $G$ | (equality + bound rows) × length of $z$ | 35 × 19 |

Index map, used in every later step:
- $x_i$ occupies `z[i*n:(i+1)*n]`;
- $u_i$ occupies `z[nx+i*m:nx+(i+1)*m]` (note the `nx` offset);
- $u_0$ therefore starts at index $(N+1)n$, which Part B uses.

### Step 2: cost → $H$ and $f$
There are no cross terms such as $x_1^TSu_0$, so $H$ is block diagonal in the order of $z$:
$$H=2\operatorname{blkdiag}(\underbrace{Q,\ldots,Q}_{x_0\ldots x_{N-1}},\ \underbrace{P_f}_{x_N},\ \underbrace{R,\ldots,R}_{u_0\ldots u_{N-1}}),\qquad f=0.$$
**The factor 2 compensates for the solver's factor 1/2.** Scalar check with $n=m=1$, $N=2$,
$z=[x_0,x_1,x_2,u_0,u_1]$: $H=2\operatorname{diag}(q,q,p,r,r)$ gives
$\tfrac12z^THz=qx_0^2+qx_1^2+px_2^2+ru_0^2+ru_1^2=J$.
The last state uses $P_f$, not $Q$. For tracking a reference,
expand $(x-r)^TQ(x-r)$ to obtain the linear term $-2Qr$; the constant can be omitted.

### Step 3: dynamics → $Ez=b$
The first equality is $x_0=\hat x_k$. Move every term of $x_{i+1}=A_dx_i+B_du_i$ to the left:
$$-A_dx_i+x_{i+1}-B_du_i=0.$$
Then write the coefficients in the column order of $z$. For the scalar system
$x_{i+1}=ax_i+bu_i$ with $N=2$:
$$\left[\begin{array}{ccc|cc}1&0&0&0&0\\-a&1&0&-b&0\\0&-a&1&0&-b\end{array}\right]
\begin{bmatrix}x_0\\x_1\\x_2\\u_0\\u_1\end{bmatrix}
=\begin{bmatrix}\hat x_k\\0\\0\end{bmatrix}.$$
For the cart-pole, replace $a$, $b$, $1$ by the blocks $A_d$ ($4\times4$), $B_d$ ($4\times1$), $I_4$.
Dynamics constraint $i$ occupies rows `(i+1)*n:(i+2)*n` (the first $n$ rows hold $x_0=\hat x_k$),
with $-A_d$ under $x_i$, $I$ under $x_{i+1}$ and $-B_d$ under $u_i$. For $N=3$:
$$E=\left[\begin{array}{cccc|ccc}I&&&&&&\\-A_d&I&&&-B_d&&\\&-A_d&I&&&-B_d&\\&&-A_d&I&&&-B_d\end{array}\right],\qquad
b=\begin{bmatrix}\hat x_k\\0\\0\\0\end{bmatrix}.$$
Blank blocks are zero.

### Step 4: box bounds
An identity matrix picks out every entry of $z$, so $-z_{\max}\le Iz\le z_{\max}$ with
$$z_{\max}=[\,\underbrace{x_{\max},\ldots,x_{\max}}_{N+1\text{ times}},\ \underbrace{u_{\max},\ldots,u_{\max}}_{N\text{ times}}\,].$$
The bounds include $x_0$. If the measured state already violates them, the QP is infeasible,
and Part B tests that you report this. Use `np.inf` to remove a box bound.

### Step 5: combine
Append the identity below $E$:
$$G=\begin{bmatrix}E\\I\end{bmatrix},\quad
l=\begin{bmatrix}b\\-z_{\max}\end{bmatrix},\quad
u=\begin{bmatrix}b\\z_{\max}\end{bmatrix}.$$
An equality is encoded with identical lower and upper bounds.
Return `(H, f, G, lower, upper)` as accepted by `solve_qp`.

### Useful tools and common mistakes
`sparse.block_diag`, `sparse.lil_matrix` (fill blocks with slices, then convert to `csc`),
`sparse.vstack`, `sparse.eye`, `np.tile` and `np.r_` are enough. $H$ must be a `scipy.sparse`
matrix, because the test calls `.toarray()`. Watch for these errors:
- $N$ states instead of $N+1$;
- forgetting the factor 2, or weighting $x_N$ with $Q$;
- wrong signs on $-A_d$ or $-B_d$;
- input columns without the `nx` offset;
- equalities written as `lower = b`, `upper = inf` instead of `lower = upper = b`.

The test simulates random inputs through the dynamics, checks that the first $(N+1)n$ rows
of $Gz$ equal `lower`, and compares $\tfrac12z^THz+f^Tz$ with $J$ summed term by term in a loop.
The matrix form and the definition must give the same number.

References: [OSQP QP form](https://osqp.org/docs/solver/index.html),
[official MPC example](https://osqp.org/docs/examples/mpc.html).

In [ ]:
def assemble_qp(A,B,Q,R,P,N,x0,xlimit,ulimit):
    """Build the sparse full-state QP in the ordering explained above."""
    # TODO
    # Step 1: sizes n, m from B; nx=(N+1)*n states, nu=N*m inputs
    # Step 2: cost H (sparse, block diagonal, times 2) and f
    # Step 3: equality block E and right-hand side b (x0 rows, then N dynamics rows)
    # Step 4: stacked bounds z_max for every state (including x0) and input
    # Step 5: G = [E; I], lower = [b; -z_max], upper = [b; z_max]
    # return H, f, G, lower, upper
    raise NotImplementedError("assemble_qp")

In [ ]:
# TEST CELL (A): ordering, dynamics, and independent cost evaluation.
n,m=4,1; N_test=3; x_test=np.array([.01,.02,0,0])
qp=assemble_qp(Ad,Bd,Q,R,P,N_test,x_test,np.full(4,np.inf),np.array([np.inf]))
H,f,G,lo,hi=qp
assert H.shape==(19,19) and G.shape==(35,19)
rng=np.random.default_rng(7); U_test=rng.normal(scale=.01,size=(N_test,1)); X_test=[x_test]
for u in U_test:X_test.append(Ad@X_test[-1]+Bd@u)
X_test=np.array(X_test);z=np.r_[X_test.ravel(),U_test.ravel()]
expected=sum(x@Q@x+u@R@u for x,u in zip(X_test[:-1],U_test))+X_test[-1]@P@X_test[-1]
np.testing.assert_allclose(.5*z@(H@z)+f@z,expected,atol=1e-10)
np.testing.assert_allclose((G@z)[:16],lo[:16],atol=1e-12)
np.testing.assert_allclose(lo[:16],hi[:16])
assert np.max(np.abs((H-H.T).toarray()))<1e-12
print('Part A tests passed')

## Part B: Apply one action, then move the window (8 pts)

Solve using `solve_qp`, extract the first control at offset $(N+1)n$, and reshape the
state block into $(N+1,n)$. Return `(first_input, predicted_states, solver_result)`.

At each simulation step call `controller(x,k)`, which returns `(u,prediction,solve_seconds)`.
Advance with `rk4(x,u,dt)`. Optional disturbance `(k,delta_x)` is added **after** that
step. Store the same fields used by the plotting helpers. Stop and record `failure`
on a QP error, non-finite state, or $|\theta|>\pi/2$; never apply an invalid solution.

Only the first input is executed. The remaining controls describe an intention, which
will be recomputed from the next observation. This is receding-horizon feedback.

In [ ]:
def mpc_action(A,B,Q,R,P,N,x,xlimit,ulimit):
    """Return first control, predicted states, and QPResult."""
    # TODO
    raise NotImplementedError("mpc_action")

In [ ]:
def rollout(controller,x0,steps,dt,disturbance=None):
    """Return X, U, predictions, seconds, failure, dt; apply one action per measurement."""
    # TODO
    raise NotImplementedError("rollout")

In [ ]:
# TEST CELL (B): DARE terminal cost reproduces unconstrained LQR.
for horizon in (1,10,25):
    u,pred,result=mpc_action(Ad,Bd,Q,R,P,horizon,x_test,np.full(4,np.inf),np.array([np.inf]))
    np.testing.assert_allclose(u,-K@x_test,atol=2e-6)
    np.testing.assert_allclose(pred[0],x_test,atol=2e-6)
    assert result.residual<2e-5
# Enforcing a bound on x0 that contradicts the measured state is infeasible.
try: mpc_action(Ad,Bd,Q,R,P,10,np.array([2.,0,0,0]),np.array([1.,1.,10.,10.]),np.array([1.]))
except RuntimeError as exc: assert 'infeasible' in str(exc).lower()
else: raise AssertionError('Infeasible QP was accepted')
seen=[]
def probe(x,k):
    seen.append(x.copy());return np.zeros(1),None,0.
r=rollout(probe,np.zeros(4),3,DT,(0,np.array([0,.01,0,0])))
np.testing.assert_allclose(seen[1],[0,.01,0,0])
assert r['X'].shape==(4,4) and r['U'].shape==(3,1)
print('Part B tests passed')

## Part C: Compare evidence, not slogans (7 pts)
Use the same plant, weights, starting state and time step. Unconstrained LQR is a
mathematical benchmark; clipped LQR shares MPC's actuator bounds but cannot anticipate
future saturation. Linear predicted constraints need not hold exactly on the nonlinear
plant. Compare costs over equal completed durations and disclose failed runs.

In [ ]:
# Given experiment driver: your functions supply all QPs and rollouts.
x0=np.array([0.,np.deg2rad(5),0.,0.]); steps=200
xlimit=np.array([.5,.35,2.,3.]); ulimit=np.array([.50])
def feedback(horizon):
    def control(x,k):
        u,pred,res=mpc_action(Ad,Bd,Q,R,P,horizon,x,xlimit,ulimit)
        return u,pred,res.seconds
    return control
runs={}
runs['LQR']=rollout(lambda x,k:(-K@x,None,0.),x0,steps,DT)
runs['Clipped LQR']=rollout(lambda x,k:(np.clip(-K@x,-ulimit,ulimit),None,0.),x0,steps,DT)
for horizon in (10,25,50):
    runs[f'MPC N={horizon}']=rollout(feedback(horizon),x0,steps,DT)
plot_comparison(runs,float(ulimit[0]));plt.show()
plot_windows(runs['MPC N=25']);plt.show()
for name,run in runs.items():print(name,metrics(run,Q,R,float(ulimit[0]),xlimit))

In [ ]:
# TEST CELL (C): actual executed limits; no assertion that MPC must beat LQR.
for name,run in runs.items():
    assert np.isfinite(run['X']).all()
    if name.startswith('MPC'):
        assert run['failure'] is None,run['failure']
        assert len(run['U'])==steps
        assert np.max(np.abs(run['U']))<=ulimit[0]+2e-5
print('Part C numerical tests passed')

In [ ]:
# Given one-shot versus re-planning experiment, with identical finite horizon.
N_open=100
H,f,G,lo,hi=assemble_qp(Ad,Bd,Q,R,P,N_open,np.zeros(4),xlimit,ulimit)
one_shot=solve_qp(H,f,G,lo,hi).z[(N_open+1)*4:].reshape(N_open,1)
kick=(20,np.array([0.,np.deg2rad(1),0.,0.]))
open_run=rollout(lambda x,k:(one_shot[k],None,0.),np.zeros(4),N_open,DT,kick)
closed_run=rollout(feedback(25),np.zeros(4),N_open,DT,kick)
plot_comparison({'One-shot plan':open_run,'Replanned MPC':closed_run},float(ulimit[0]));plt.show()
if os.environ.get('COURSE_HEADLESS','0')!='1':
    display(HTML(animate_cartpole(runs['MPC N=25']).to_jshtml()))

### Short answers (Part C)
1. Why does a DARE terminal weight make unconstrained MPC reproduce LQR here?
2. Why is applying an entire precomputed input sequence not receding-horizon feedback?
3. Which constraints became active? Compare predicted and measured violations.
4. What changed with horizon length, and what computational cost did it introduce?
5. Why do a feasible linear QP and successful rollout not prove global nonlinear stability?

**Write your answers here.**

## Part D: Soft state bounds (5 bonus)
Introduce $\epsilon_i\ge0$ so $-x_{\max}-\epsilon_i\le x_i\le x_{\max}+\epsilon_i$.
Add $\rho\|\epsilon\|^2$ to the cost, but keep dynamics and actuator limits hard.
Append one slack per state coordinate, including $x_0$, and return the augmented QP.
A positive slack reports a violation; it does not make the physical limit satisfied.

In [ ]:
def soften_states(qp,n,N,weight=1e3):
    """Append nonnegative state slacks; keep input/dynamics constraints hard."""
    # TODO
    raise NotImplementedError("soften_states")

In [ ]:
# TEST CELL (D, BONUS)
small=assemble_qp(Ad,Bd,Q,R,P,5,np.array([.02,0,0,0]),np.array([.01,1,10,10]),np.array([.1]))
soft=solve_qp(*soften_states(small,4,5))
assert soft.z[small[0].shape[0]]>=.01-2e-5
assert np.max(np.abs(soft.z[24:29]))<=.1+2e-5
print('Bonus tests passed; positive slack correctly reports a state-limit violation')

## Submission
Restart the kernel and run all required sections. Submit the notebook and PDF report;
bonus is optional. Include comparison plots, numerical metrics, and the short answers.
Do not submit teacher files. See README for environment and headless validation.